In [23]:
import os
import pandas as pd
import time
from matplotlib import pyplot as plt
import seaborn as sns
from scr.qsvdd_core.data_loader import QuantumDataLoader
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from scripts.train_model import circuit_training, train_five_times
from scr.qsvdd_core.data_loader import QuantumDataLoader
from scripts.test_model import test, mean_auc, best_batch, save_test_results

In [24]:
np.random.seed(42)

In [25]:
n_train = 0 ; latent_dim = 3
# num_params_conv = 375
cost_func = 'svdd'
learning_rate = 0.01

# Breast Cancer anomaly Detection

In [26]:
print("="*60)
print("Loading and processing the Breast Cancer dataset...")
print("="*60)

# 1. Fetch the dataset
breast_cancer = fetch_ucirepo(id=17)
X_bc_raw = breast_cancer.data.features
y_bc_raw = breast_cancer.data.targets

Loading and processing the Breast Cancer dataset...


In [39]:
print(breast_cancer.data.features.head())

   radius1  texture1  perimeter1   area1  smoothness1  compactness1  \
0    17.99     10.38      122.80  1001.0      0.11840       0.27760   
1    20.57     17.77      132.90  1326.0      0.08474       0.07864   
2    19.69     21.25      130.00  1203.0      0.10960       0.15990   
3    11.42     20.38       77.58   386.1      0.14250       0.28390   
4    20.29     14.34      135.10  1297.0      0.10030       0.13280   

   concavity1  concave_points1  symmetry1  fractal_dimension1  ...  radius3  \
0      0.3001          0.14710     0.2419             0.07871  ...    25.38   
1      0.0869          0.07017     0.1812             0.05667  ...    24.99   
2      0.1974          0.12790     0.2069             0.05999  ...    23.57   
3      0.2414          0.10520     0.2597             0.09744  ...    14.91   
4      0.1980          0.10430     0.1809             0.05883  ...    22.54   

   texture3  perimeter3   area3  smoothness3  compactness3  concavity3  \
0     17.33      184.60 

In [40]:
X_bc_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   radius1             569 non-null    float64
 1   texture1            569 non-null    float64
 2   perimeter1          569 non-null    float64
 3   area1               569 non-null    float64
 4   smoothness1         569 non-null    float64
 5   compactness1        569 non-null    float64
 6   concavity1          569 non-null    float64
 7   concave_points1     569 non-null    float64
 8   symmetry1           569 non-null    float64
 9   fractal_dimension1  569 non-null    float64
 10  radius2             569 non-null    float64
 11  texture2            569 non-null    float64
 12  perimeter2          569 non-null    float64
 13  area2               569 non-null    float64
 14  smoothness2         569 non-null    float64
 15  compactness2        569 non-null    float64
 16  concavity2         

In [41]:
y_bc_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Diagnosis  569 non-null    str  
dtypes: str(1)
memory usage: 4.6 KB


In [27]:
# 2. Binarization for Anomaly Detection
# 'M' (Malignant) = 1 (Anomaly), 'B' (Benign) = 0 (Normal)
y_bc = y_bc_raw.iloc[:, 0].apply(lambda x: 1 if x == 'M' else 0).values

# 3. Use the DataLoader for classical scaling (StandardScaler)
loader = QuantumDataLoader()
# We pass the raw features and the binarized targets
# Note: Since prepare_classic_data expects a DataFrame with a "Class" column,
# we temporarily combine them or adjust the method.
import pandas as pd
df_bc = pd.DataFrame(X_bc_raw)
df_bc['Class'] = y_bc

X_data = loader.prepare_bc_data(X_bc_raw)
y = y_bc

# 4. Identify indices for each class
normal_indices = np.where(y == 0)[0]
abnormal_indices = np.where(y == 1)[0]
np.random.seed(42)

# 5. Training Set (One-Class: 200 normal samples)
X_train_normal_indices = np.random.choice(normal_indices, 250, replace=False)
X_train = X_data[X_train_normal_indices]
Y_train = y[X_train_normal_indices]

# 6. Test Set (Balanced: 50 normal + 50 anomalies)
remaining_normal_indices = list(set(normal_indices) - set(X_train_normal_indices))

X_test_normal_indices = np.random.choice(remaining_normal_indices, 50, replace=False)
X_test_abnormal_indices = np.random.choice(abnormal_indices, 50, replace=False)

X_test_normal = X_data[X_test_normal_indices]
y_test_normal = y[X_test_normal_indices]

X_test_abnormal = X_data[X_test_abnormal_indices]
y_test_abnormal = y[X_test_abnormal_indices]

# Final test set (Mix: 100 samples)
X_test = np.concatenate((X_test_normal, X_test_abnormal), axis=0)
Y_test = np.concatenate((y_test_normal, y_test_abnormal), axis=0)

print(f"X_train shape (Normal): {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"X_test shape (Mix): {X_test.shape}")
print(f"Y_test shape: {Y_test.shape}")

X_train shape (Normal): (250, 32)
Y_train shape: (250,)
X_test shape (Mix): (100, 32)
Y_test shape: (100,)


In [28]:
loader = QuantumDataLoader()

X_quantum = loader.prepare_bc_data(X_bc_raw)

print(f"Features for the circuit: {X_quantum.shape}")
print(f"Labels: {y.shape}")

Features for the circuit: (569, 32)
Labels: (569,)


In [29]:
"""
One-class Training:
Separation into normal and fraudulent examples
QSVDD will learn what is normal.
"""

normal_indices = np.where(y == 0)[0]
abnormal_indices = np.where(y == 1)[0]
np.random.seed(42)

# Train
# collect 1000 normal examples for training
X_train_normal_indices = np.random.choice(normal_indices, 250, replace=False)
X_train_normal = X_quantum[X_train_normal_indices]
y_train_normal = y[X_train_normal_indices]

# X_train contains only legitimate transactions
X_train = X_train_normal
Y_train = y_train_normal

# Test (balanced)
# The code removes 100 normal examples that were not used in training
remaining_normal_indices = list(set(normal_indices) - set(X_train_normal_indices))
X_test_normal_indices = np.random.choice(remaining_normal_indices, 100, replace=False)
X_test_normal = X_quantum[X_test_normal_indices]
y_test_normal = y[X_test_normal_indices]

# The code removes 100 fraud examples.
X_test_abnormal_indices = np.random.choice(abnormal_indices, 100, replace=False)
X_test_abnormal = X_quantum[X_test_abnormal_indices]
y_test_abnormal = y[X_test_abnormal_indices]

# creates a test set with 200 examples (50% normal, 50% fraud)
X_test = np.concatenate((X_test_normal, X_test_abnormal), axis=0)
Y_test = np.concatenate((y_test_normal, y_test_abnormal), axis=0)

center = np.zeros(latent_dim)
center_train = np.tile(center, (len(X_train), 1))

print(f'X_train shape: {X_train.shape}')
print(f'Y_train shape: {Y_train.shape}')
print(f'X_test_normal shape: {X_test_normal.shape}')
print(f'X_test_abnormal shape: {X_test_abnormal.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Y_test shape: {Y_test.shape}')
print(f'center_train shape: {center_train.shape}')

X_train shape: (250, 32)
Y_train shape: (250,)
X_test_normal shape: (100, 32)
X_test_abnormal shape: (100, 32)
X_test shape: (200, 32)
Y_test shape: (200,)
center_train shape: (250, 3)


In [30]:
train_Xdata = X_train
train_Ydata = center_train

### QCNN (Quantum Convolutional Neural Network) Ansatz

#### Hyperparameters

In [31]:
qcnn_batch_size = 4
qcnn_steps = 2000

#### Single Training Run

In [22]:
qcnn_start_time = time.time()
qcnn_loss_history, qcnn_est_params, qcnn_param_history = (
    circuit_training(X_train=train_Xdata,
                     Y_train=train_Ydata,
                     batch_size=qcnn_batch_size,
                     learning_rate=learning_rate,
                     steps=qcnn_steps,
                     ansatz='qcnn'
                     )
)
qcnn_end_time = time.time()

qcnn_total_runtime = (qcnn_end_time - qcnn_start_time) / 60
print(f"Total runtime: {qcnn_total_runtime} minutes for batch size {qcnn_batch_size}")

/home/jvfg/Documents/ORG/Repos/QSVDD2/.venv/lib/python3.12/site-packages/autograd/numpy/numpy_vjps.py:943: ComplexWarning: Casting complex values to real discards the imaginary part
  onp.add.at(A, idx, x)


Total runtime: 0.13480319579442343 minutes for batch size 4


In [33]:
(qcnn_loss_history_matrix,
 qcnn_est_params_matrix,
 qcnn_param_history_matrix,
 qcnn_time_record) = train_five_times(X_train=train_Xdata,
                                        Y_train=train_Ydata,
                                        batch_size=qcnn_batch_size,
                                        learning_rate=learning_rate,
                                        steps=qcnn_steps,
                                        ansatz='qcnn'
                                        )
loss_history_f_name = f"../results/training/QCNN/BC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{learning_rate:.0e}_LOSS_HISTORY_MEAN.npy"
est_params_f_name = f"../results/training/QCNN/BC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{learning_rate:.0e}_EST_PARAMS_MEAN.npy"
time_f_name = f"../results/training/QCNN/BC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{learning_rate:.0e}_TIME_MEAN.npy"
np.savetxt(loss_history_f_name, qcnn_loss_history_matrix)
np.savetxt(est_params_f_name, qcnn_est_params_matrix)
np.savetxt(time_f_name, qcnn_time_record)
print("--- All training batches completed ---")

--- Starting training round 1 with seed 1848857951 ---


/home/jvfg/Documents/ORG/Repos/QSVDD2/.venv/lib/python3.12/site-packages/autograd/numpy/numpy_vjps.py:943: ComplexWarning: Casting complex values to real discards the imaginary part
  onp.add.at(A, idx, x)


--- Starting training round 2 with seed 452643366 ---
--- Starting training round 3 with seed 3619198621 ---
--- Starting training round 4 with seed 1328972624 ---
--- Starting training round 5 with seed 919471137 ---
--- All training batches completed ---


#### QCNN Training Evaluation

In [ ]:
f_name = f"../results/training/QCNN/BC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{learning_rate:.0e}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="qcnn")

print(50*"--")
print(f'for: B{qcnn_batch_size}S{qcnn_steps} | AUC_mean: {mean} | std: {std}')
print(50*"--")

#### Saving Noiseless Training with QCNN ansatz

In [ ]:
np.savetxt(f"../results/training/QCNN/QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}_LOSS_HISTORY.npy", qcnn_loss_history)
np.savetxt(f"../results/training/QCNN/QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}_EST_PARAMS.npy", qcnn_est_params)
np.savetxt(f"../results/training/QCNN/QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}_PARAM_HISTORY.npy", qcnn_param_history)

### QAE (Quantum AutoEncoder) Ansatz

In [ ]:
qae_batch_size = 16
qae_steps = 500

#### Five-Run Training & Result Persistence

Run `train_five_times` to train the **QAE** ansatz across 5 independent runs with the configured batch size and step count.

In [ ]:
(qae_loss_history_matrix,
 qae_est_params_matrix,
 qae_param_history_matrix,
 qae_time_record) = train_five_times(X_train=train_Xdata,
                                        Y_train=train_Ydata,
                                        batch_size=qae_batch_size,
                                        learning_rate=learning_rate,
                                        steps=qae_steps,
                                        ansatz='qae'
                                        )
loss_history_f_name = f"../results/training/QAE/BC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{learning_rate:.0e}_LOSS_HISTORY_MEAN.npy"
est_params_f_name = f"../results/training/QAE/BC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{learning_rate:.0e}_EST_PARAMS_MEAN.npy"
time_f_name = f"../results/training/QAE/BC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{learning_rate:.0e}_TIME_MEAN.npy"
np.savetxt(loss_history_f_name, qae_loss_history_matrix)
np.savetxt(est_params_f_name, qae_est_params_matrix)
np.savetxt(time_f_name, qae_time_record)
print("--- All training batches completed ---")

#### QAE Training Evaluation

In [ ]:
f_name = f"../results/training/QAE/QAE_B{qae_batch_size:02d}S{qae_steps}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="qae")

print(50*"--")
print(f'for: B{qae_batch_size}S{qae_steps} | AUC_mean: {mean} | std: {std}')
print(50*"--")

#### Saving noiseless training with QAE Ansatz

In [ ]:
np.savetxt(f"../results/training/QAE/BC_QAE_B{qae_batch_size:02d}S{qae_steps}_LOSS_HISTORY.npy", qae_loss_history)
np.savetxt(f"../results/training/QAE/BC_QAE_B{qae_batch_size:02d}S{qae_steps}_EST_PARAMS.npy", qae_est_params)
np.savetxt(f"../results/training/QAE/BC_QAE_B{qae_batch_size:02d}S{qae_steps}_PARAM_HISTORY.npy", qae_param_history)

### LCQHNN (Lean classical-quantum hybrid neural network) Ansatz

The latent space for this ansatz has dimension 5 (vs. 3 for QCNN/QAE), requiring a re-initialized center vector.

In [ ]:
center = np.zeros(5)
center_train = np.tile(center, (len(X_train), 1))
lcqhnn_batch_size = 8
lcqhnn_steps = 1000
print(f'center_train shape: {center_train.shape}')
train_Xdata = X_train
train_Ydata = center_train